In [ ]:
import os
import h5py
import copy
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm import tqdm
import matplotlib.pyplot as plt
from skimage.transform import radon
from scipy.optimize import curve_fit
from scipy.special import erf, erfinv
from matplotlib.patches import Polygon
from mpl_toolkits.axes_grid1 import make_axes_locatable
from tools import analyse_waveforms, save_analysis, read_analysis, H5FileManager

In [ ]:
sample_name = "TW5_V1"
wafer_type = "W5"
data_folder = "/media/tct/T7/ti-LGADs"

In [ ]:
# def check_150v_for_nans(filename):
#     target_v = "voltage_0150V"
#     print(f"--- Fast Scan: {filename} ({target_v}) ---")

#     with h5py.File(filename, "r") as f:
#         if target_v not in f:
#             print(f" Group {target_v} not found in the file.")
#             return

#         v_grp = f[target_v]
#         positions = [
#             name for name in v_grp.keys() if isinstance(v_grp[name], h5py.Group)
#         ]

#         corrupted_positions = []

#         for pos_name in tqdm(positions, desc=f"Scanning {target_v}", unit="pos"):
#             pos_grp = v_grp[pos_name]

#             for ds_name in pos_grp.keys():
#                 obj = pos_grp[ds_name]

#                 if isinstance(obj, h5py.Dataset):
#                     data = obj[:]

#                     if np.isnan(data).any():
#                         nan_rows = np.where(np.isnan(data).any(axis=1))[0]
#                         tqdm.write(
#                             f"NaN in {pos_name} -> {ds_name} | Triggers: {nan_rows.tolist()}"
#                         )
#                         corrupted_positions.append(pos_name)

#         print("\n" + "=" * 30)
#         if not corrupted_positions:
#             print(f" Status: Group {target_v} is clean.")
#         else:
#             unique_bad = len(set(corrupted_positions))
#             print(f" Alert: Found {unique_bad} corrupted positions in {target_v}.")

In [ ]:
# filename = f"{data_folder}/{wafer_type}/{sample_name}/{sample_name}.h5"
# check_150v_for_nans(filename)

In [ ]:
# voltages = []

In [ ]:
# for voltage in voltages:
#     df = analyse_waveforms(
#         f"{data_folder}/{wafer_type}/{sample_name}/{sample_name}.h5", voltage
#     )
#     save_analysis(
#         df, f"{data_folder}/{wafer_type}/{sample_name}/{sample_name}_{voltage}V.csv"
#     )

In [ ]:
# save_analysis(df, f"{data_folder}/{wafer_type}/{sample_name}/{sample_name}_{voltage}V.csv")

In [ ]:
# H5FileManager.split_by_voltage(f"{data_folder}/{wafer_type}/{sample_name}/{sample_name}.h5")
# H5FileManager.merge_files(f"{data_folder}/{wafer_type}/{sample_name}/combined_results.h5", files_to_combine)

In [ ]:
def gauss(x, amp, cen, wid):
    return amp * np.exp(-((x - cen) ** 2) / 2 / wid / wid)

def sigmoid(x, amp, cen, wid):
    return amp / (1 + np.exp(-1 * (x - cen) / wid))

def sigmoid_inverse(x, amp, cen, wid):
    return cen - wid * np.log(amp / x - 1)

def sigmoid_inverse_error(x, amp, cen, wid, d_amp, d_cen, d_wid):
    return np.sqrt(
        d_cen**2
        + np.log(amp / x - 1) * np.log(amp / x - 1) * d_wid**2
        + wid**2 / (amp / x - 1) / (amp / x - 1) * d_amp**2 / x / x
    )

def erf_edge(x, A, mu, sigma, sign=1):
    return (A / 2) * (1 + sign * erf((x - mu) / (np.sqrt(2) * sigma)))

def erf_inv_edge(th, mu, sigma, sign=1):
    return mu + np.sqrt(2) * sigma * sign * erfinv(2 * th - 1)

def erf_inv_err_edge(th, err_mu, err_sigma, sign=1):
    k = np.sqrt(2) * sign * erfinv(2 * th - 1)
    return np.sqrt(err_mu**2 + (k * err_sigma)**2)

In [ ]:
def to_plot_coords(x_raw, y_raw, x_vals, y_vals):
    px = np.interp(x_raw, x_vals, np.arange(len(x_vals)) + 0.5)
    py = np.interp(y_raw, y_vals, np.arange(len(y_vals)) + 0.5)
    return px, py


def get_intersection(line1, line2):
    k1, b1 = line1["k"], line1["b"]
    k2, b2 = line2["k"], line2["b"]
    x_int = (b2 - b1) / (k1 - k2)
    y_int = k1 * x_int + b1
    return x_int, y_int


def plot_amplitude_map(df, voltage, borders=None, interpad_lines=None):
    agg_df = (
        df[df["pulse_number"] == 1]
        .groupby(["x", "y"], as_index=False)["peak_integral"]
        .sum()
    )
    pivot = agg_df.pivot(index="y", columns="x", values="peak_integral")
    pivot = pivot.sort_index().sort_index(axis=1)

    x_vals = pivot.columns.values
    y_vals = pivot.index.values

    dx = np.abs(np.median(np.diff(x_vals)))
    dy = np.abs(np.median(np.diff(y_vals)))

    _, ax = plt.subplots(figsize=(10, 10))

    sns.heatmap(
        np.abs(pivot),
        cmap="plasma",
        cbar_kws={"label": "Sum of Amplitudes [V]"},
        ax=ax,
    )

    if pd.notna(dx) and pd.notna(dy) and dx > 0:
        ax.set_aspect(dy / dx)

    ax.set_title(f"TCT Scan - {voltage} V - Combined Channels")
    ax.set_xlabel(r"x ($\mu$m)")
    ax.set_ylabel(r"y ($\mu$m)")


    tick_spacing = 1

    ax.set_xticks(np.arange(len(x_vals))[::tick_spacing] + 0.5)
    ax.set_xticklabels(x_vals[::tick_spacing].astype(int), rotation=45)

    ax.set_yticks(np.arange(len(y_vals))[::tick_spacing] + 0.5)
    ax.set_yticklabels(y_vals[::tick_spacing].astype(int), rotation=0)

    if borders is not None:

        corners_raw = [
            get_intersection(borders["left"], borders["bottom"]),
            get_intersection(borders["right"], borders["bottom"]),
            get_intersection(borders["right"], borders["top"]),
            get_intersection(borders["left"], borders["top"]),
        ]

        pixels = [to_plot_coords(x, y, x_vals, y_vals) for x, y in corners_raw]

        rect = Polygon(
            pixels,
            closed=True,
            linewidth=2,
            edgecolor="black",
            facecolor="none",
        )
        ax.add_patch(rect)

        if interpad_lines is not None:
            for _, line_data in interpad_lines.items():
                p1_raw = get_intersection(borders["left"], line_data)
                p2_raw = get_intersection(borders["right"], line_data)

                px1, py1 = to_plot_coords(*p1_raw, x_vals, y_vals)
                px2, py2 = to_plot_coords(*p2_raw, x_vals, y_vals)

                ax.plot(
                    [px1, px2], [py1, py2], color="black", linewidth=2, linestyle="-"
                )

    plt.tight_layout()
    plt.show()

In [ ]:
voltage = 150

df = read_analysis(
    f"{data_folder}/{wafer_type}/{sample_name}/{sample_name}_{voltage}V.csv"
)

plot_amplitude_map(df, voltage)

In [ ]:
def get_sensor_borders(df, threshold_fraction=0.5):
    df_p1 = df[df["pulse_number"] == 1].copy()

    pivot = df_p1.pivot_table(
        index="y", columns="x", values="peak_amplitude", aggfunc="mean"
    ).fillna(0)
    image = np.abs(pivot.to_numpy())

    theta = np.linspace(-90.0, 90.0, 180, endpoint=False)
    sinogram = radon(image, theta=theta, circle=False)

    angle_deg = theta[np.argmax(np.var(sinogram, axis=0))]
    angle_rad = np.deg2rad(angle_deg)
    cos_a, sin_a = np.cos(angle_rad), np.sin(angle_rad)

    df_p1["u"] = df_p1["x"] * cos_a - df_p1["y"] * sin_a
    df_p1["v"] = df_p1["x"] * sin_a + df_p1["y"] * cos_a

    u_mins, u_maxs = [], []
    v_mins, v_maxs = [], []

    for ch in df_p1["ch"].unique():
        df_ch = df_p1[df_p1["ch"] == ch]

        u_prof = df_ch.groupby("u")["peak_amplitude"].sum().abs().reset_index()
        u_max_val = u_prof["peak_amplitude"].max()
        u_mask = u_prof["peak_amplitude"] >= (u_max_val * threshold_fraction)
        if u_mask.any():
            u_mins.append(u_prof["u"][u_mask].min())
            u_maxs.append(u_prof["u"][u_mask].max())

        v_prof = df_ch.groupby("v")["peak_amplitude"].sum().abs().reset_index()
        v_max_val = v_prof["peak_amplitude"].max()
        v_mask = v_prof["peak_amplitude"] >= (v_max_val * threshold_fraction)
        if v_mask.any():
            v_mins.append(v_prof["v"][v_mask].min())
            v_maxs.append(v_prof["v"][v_mask].max())

    u_min_global = np.mean(u_mins)
    u_max_global = np.mean(u_maxs)

    v_min_global = np.min(v_mins)
    v_max_global = np.max(v_maxs)

    eps = 1e-12
    safe_sin = sin_a if abs(sin_a) > eps else eps
    safe_cos = cos_a if abs(cos_a) > eps else eps

    borders = {}

    k_lr = cos_a / safe_sin
    borders["left"] = {"k": k_lr, "b": -u_min_global / safe_sin}
    borders["right"] = {"k": k_lr, "b": -u_max_global / safe_sin}

    k_tb = -sin_a / safe_cos
    borders["bottom"] = {"k": k_tb, "b": v_min_global / safe_cos}
    borders["top"] = {"k": k_tb, "b": v_max_global / safe_cos}

    return borders, angle_deg

In [ ]:
borders, angle_deg = get_sensor_borders(df, threshold_fraction=0.7)
print("Estimated borders (y = kx + b):")
for key, params in borders.items():
    print(f" {key.capitalize()}: k = {params['k']:.4f}, b = {params['b']:.2f}")

In [ ]:
plot_amplitude_map(df, voltage, borders)

In [ ]:
def analyze_cce(df, voltage, angle_deg, threshold=0.9, make_plot=False, normalization=False):
    y_borders_internal = {}
    y_borders_err = {}
    fit_popt_pcov = {}

    angle_rad = np.deg2rad(angle_deg)
    cos_a, sin_a = np.cos(angle_rad), np.sin(angle_rad)

    dy = np.abs(np.median(np.diff(np.sort(df["y"].unique()))))
    if pd.isna(dy) or dy == 0:
        dy = 10.0

    if make_plot:
        plt.figure(figsize=(10, 6))

    channels = sorted(df["ch"].unique())

    for ch in channels:
        df_ch = df[(df["ch"] == ch) & (df["pulse_number"] == 1)].copy()

        df_ch["v"] = df_ch["x"] * sin_a + df_ch["y"] * cos_a

        df_ch["v_bin"] = np.round(df_ch["v"] / dy) * dy
        agg_df = (
            df_ch.groupby("v_bin", as_index=False)["peak_integral"]
            .sum()
            .sort_values("v_bin")
        )

        v_arr = agg_df["v_bin"].values
        amp_arr = np.abs(agg_df["peak_integral"].values)
        max_amp_raw = np.max(amp_arr)
        plateau_val = np.median(amp_arr[amp_arr > 0.8 * max_amp_raw]) if max_amp_raw > 0 else 1.0
        
        if normalization:
            plateau_val = np.median(amp_arr[amp_arr > 0.8 * max_amp_raw]) if max_amp_raw > 0 else 1.0
            amp_arr = amp_arr / plateau_val
            
        if make_plot:
            plt.plot(v_arr, amp_arr, marker="o", label=f"Ch {ch}")

        max_amp = np.max(amp_arr)
        plateau_mask = amp_arr > (0.5 * max_amp)
        if not np.any(plateau_mask):
            continue

        pad_center = np.median(v_arr[plateau_mask])
        l_mask, r_mask = v_arr <= pad_center, v_arr > pad_center

        if not np.any(l_mask) or not np.any(r_mask):
            continue

        try:
            mu_guess_l = v_arr[l_mask][np.argmin(np.abs(amp_arr[l_mask] - 0.5 * max_amp))]
            p_l, c_l = curve_fit(
                lambda x, A, mu, sigma: erf_edge(x, A, mu, sigma, sign=1),
                v_arr[l_mask],
                amp_arr[l_mask],
                p0=[max_amp, mu_guess_l, 10.0],
                maxfev=10000,
            )
      
            mu_guess_r = v_arr[r_mask][np.argmin(np.abs(amp_arr[r_mask] - 0.5 * max_amp))]
            p_r, c_r = curve_fit(
                lambda x, A, mu, sigma: erf_edge(x, A, mu, sigma, sign=-1),
                v_arr[r_mask],
                amp_arr[r_mask],
                p0=[max_amp, mu_guess_r, 10.0],
                maxfev=10000,
            )

            err_mu_l, err_sig_l = np.sqrt(c_l[1, 1]), np.sqrt(c_l[2, 2])
            err_mu_r, err_sig_r = np.sqrt(c_r[1, 1]), np.sqrt(c_r[2, 2])

            b_l = erf_inv_edge(threshold, p_l[1], abs(p_l[2]), sign=1)
            e_l = erf_inv_err_edge(threshold, err_mu_l, err_sig_l, sign=1)
            
            b_r = erf_inv_edge(threshold, p_r[1], abs(p_r[2]), sign=-1)
            e_r = erf_inv_err_edge(threshold, err_mu_r, err_sig_r, sign=-1)

            y_borders_internal[ch] = [b_l, b_r]
            y_borders_err[ch] = [e_l, e_r]
            fit_popt_pcov[ch] = {"left": (p_l, c_l), "right": (p_r, c_r)}

            if make_plot:
                plt.axvline(b_l, color="green", linestyle="--", alpha=0.5)
                plt.axvline(b_r, color="black", linestyle="--", alpha=0.5)
                         
        except Exception as e:
            print(f"Fit failed for channel {ch}: {e}")
            continue

    if len(y_borders_internal) < 2:
        return {}, 0, 0

    ch_ids = list(y_borders_internal.keys())
    centers = {c: np.mean(y_borders_internal[c]) for c in ch_ids}
    bottom_ch = min(centers, key=centers.get)
    top_ch = max(centers, key=centers.get)

    inner_b = y_borders_internal[bottom_ch][1]
    inner_t = y_borders_internal[top_ch][0]

    cce_interpad_distance = inner_t - inner_b

    e_b = y_borders_err[bottom_ch][1]
    e_t = y_borders_err[top_ch][0]
    cce_interpad_distance_error = np.sqrt(e_b**2 + e_t**2)

    eps = 1e-12
    safe_cos = cos_a if abs(cos_a) > eps else eps
    k_line = -sin_a / safe_cos

    cce_interpad_lines = {
        "inner_bottom": {"k": k_line, "b": inner_b / safe_cos},
        "inner_top": {"k": k_line, "b": inner_t / safe_cos},
    }

    if make_plot:
        norm_str = " (Normalized)" if normalization else ""
        plt.title(f"TCT Scan - {voltage} V - Y Profile (CCE){norm_str}")
        plt.xlabel(r"Projected V (Height) [$\mu$m]")
        
        y_label = "Normalized Charge [a.u.]" if normalization else r"Total sum of impedance-charge [V$\cdot$ns]"
        plt.ylabel(y_label)

        textstr = rf"CCE Gap: {cce_interpad_distance:.1f} $\pm$ {cce_interpad_distance_error:.1f} $\mu$m, {threshold*100:.0f}% threshold"
        plt.text(
            0.5,
            0.15,
            textstr,
            transform=plt.gca().transAxes,
            fontsize=12,
            ha="center",
            va="center",
            bbox=dict(facecolor="white", alpha=0.8),
        )

        plt.legend(loc="upper left")
        plt.grid(True, alpha=0.5)
        plt.tight_layout()
        plt.show()

    return cce_interpad_lines, cce_interpad_distance, cce_interpad_distance_error

In [ ]:
cce_interpad_lines, cce_interpad_distance, cce_interpad_distance_error = analyze_cce(
    df, voltage, angle_deg, threshold=0.9, make_plot=True, normalization=True
)

In [ ]:
def analyze_cs(df, voltage, angle_deg, threshold=0.9, make_plot=True):
    angle_rad = np.deg2rad(angle_deg)
    cos_a, sin_a = np.cos(angle_rad), np.sin(angle_rad)

    dy = np.abs(np.median(np.diff(np.sort(df["y"].unique()))))
    if pd.isna(dy) or dy == 0: 
        dy = 10.0

    channels = sorted(df["ch"].unique())
    channel_data = {}

    for ch in channels:
        df_ch = df[(df["ch"] == ch) & (df["pulse_number"] == 1)].copy()
        df_ch["v"] = df_ch["x"] * sin_a + df_ch["y"] * cos_a
        df_ch["v_bin"] = np.round(df_ch["v"] / dy) * dy
        
        agg_df = df_ch.groupby("v_bin", as_index=False)["peak_integral"].sum().sort_values("v_bin")
        channel_data[ch] = {"v": agg_df["v_bin"].values, "amp": np.abs(agg_df["peak_integral"].values)}

    if len(channel_data) < 2: 
        return np.nan, np.nan, np.nan, {}

    ch_ids = list(channel_data.keys())
    centers = {c: np.average(channel_data[c]["v"], weights=channel_data[c]["amp"]) for c in ch_ids}
    bottom_ch = min(centers, key=centers.get)
    top_ch = max(centers, key=centers.get)

    v_min = max(channel_data[bottom_ch]["v"].min(), channel_data[top_ch]["v"].min())
    v_max = min(channel_data[bottom_ch]["v"].max(), channel_data[top_ch]["v"].max())
    v_common = np.arange(v_min, v_max + dy, dy)

    amp_b_raw = np.interp(v_common, channel_data[bottom_ch]["v"], channel_data[bottom_ch]["amp"])
    amp_t_raw = np.interp(v_common, channel_data[top_ch]["v"], channel_data[top_ch]["amp"])

    max_b = np.max(amp_b_raw)
    plateau_b = np.median(amp_b_raw[amp_b_raw > 0.8 * max_b]) if max_b > 0 else 1.0

    max_t = np.max(amp_t_raw)
    plateau_t = np.median(amp_t_raw[amp_t_raw > 0.8 * max_t]) if max_t > 0 else 1.0
    
    min_plateau = min(plateau_b, plateau_t)
    cs_asymmetry = (abs((plateau_t - plateau_b)) / min_plateau) * 100.0

    amp_b_norm = np.clip(amp_b_raw / plateau_b, 0, None)
    amp_t_norm = np.clip(amp_t_raw / plateau_t, 0, None)
    
    sum_norm_eta = amp_b_norm + amp_t_norm
    eta = np.full_like(sum_norm_eta, np.nan)
    
    valid_mask = sum_norm_eta > 0.05
    eta[valid_mask] = amp_t_norm[valid_mask] / sum_norm_eta[valid_mask]

    fit_success = False
    mu_eta, sig_eta, cs_width = np.nan, np.nan, np.nan
    mu_eta_err, sig_eta_err, cs_width_error = np.nan, np.nan, np.nan
    w_left, w_right = np.nan, np.nan
    
    try:
        if np.sum(valid_mask) > 5:
            valid_v = v_common[valid_mask]
            valid_eta = eta[valid_mask]
            cross_idx = np.argmin(np.abs(valid_eta - 0.5))
            mu_guess = valid_v[cross_idx]
            
            popt_eta, pcov_eta = curve_fit(lambda x, mu, sig: erf_edge(x, 1, mu, sig), valid_v, valid_eta, p0=[mu_guess, 5.0], maxfev=10000)
            mu_eta, sig_eta = popt_eta[0], abs(popt_eta[1])
            perr_eta = np.sqrt(np.diag(pcov_eta))
            mu_eta_err, sig_eta_err = perr_eta[0], perr_eta[1]
            
            fit_success = True
            
            width_factor = np.sqrt(2) * (erfinv(2*threshold - 1) - erfinv(2*(1-threshold) - 1))
            cs_width = width_factor * sig_eta
            cs_width_error = width_factor * sig_eta_err
            
            w_left = mu_eta - cs_width / 2
            w_right = mu_eta + cs_width / 2
    except Exception as e:
        print(f"Eta fit failed: {e}")

    cs_interpad_lines = {}
    if fit_success and not np.isnan(w_left) and not np.isnan(w_right):
        v_bot = min(w_left, w_right)
        v_top = max(w_left, w_right)
        
        eps = 1e-12
        safe_cos = cos_a if abs(cos_a) > eps else eps
        k_line = -sin_a / safe_cos
        
        cs_interpad_lines = {
            'inner_bottom': {'k': k_line, 'b': v_bot / safe_cos},
            'inner_top': {'k': k_line, 'b': v_top / safe_cos}
        }

    if make_plot:
        _, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 10), sharex=True)
        colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
        c_b, c_t = colors[0], colors[1]

        ax1.plot(v_common, amp_b_raw, marker=".", linestyle="-", color=c_b, alpha=0.5, label=f"Ch {bottom_ch} (Raw)")
        ax1.plot(v_common, amp_t_raw, marker=".", linestyle="-", color=c_t, alpha=0.5, label=f"Ch {top_ch} (Raw)")
        ax1.plot(v_common, amp_b_raw + amp_t_raw, marker="o", linestyle="-", color="purple", linewidth=2.5, label="Total Raw Charge")
        
        ax1.axhline(plateau_b, color=c_b, linestyle="--", alpha=0.8, linewidth=2)
        ax1.axhline(plateau_t, color=c_t, linestyle="--", alpha=0.8, linewidth=2)

        asym_str = (f"Ch {bottom_ch} Plateau: {plateau_b:.1f} V·ns\n"
                    f"Ch {top_ch} Plateau: {plateau_t:.1f} V·ns\n"
                    f"Asymmetry: {cs_asymmetry:+.1f}%")
        
        ax1.text(0.02, 0.85, asym_str, transform=ax1.transAxes, fontsize=11,
                 bbox=dict(facecolor="white", alpha=0.9, edgecolor="gray"))

        ax1.set_ylabel(r"Absolute Charge [V$\cdot$ns]")
        ax1.set_title(f"TCT Scan - {voltage} V - Charge Sharing & Asymmetry")
        ax1.grid(True, alpha=0.5)
        ax1.legend(loc="upper right")

        if np.sum(valid_mask) > 0:
            ax2.plot(v_common[valid_mask], eta[valid_mask], marker="o", linestyle="", color="black", alpha=0.6, label=r"Measured $\eta$")
        
        if fit_success:
            v_smooth = np.linspace(v_common.min(), v_common.max(), 300)
            ax2.plot(v_smooth, erf_edge(v_smooth, 1, mu_eta, sig_eta), color="red", linewidth=2.5, label=r"Fitted $\eta$")
            
            ax2.axvline(w_left, color="darkorange", linestyle=":", linewidth=2)
            ax2.axvline(w_right, color="darkorange", linestyle=":", linewidth=2)
            
            ax2.axhline(0.5, color="gray", linestyle=":", alpha=0.7)

            eta_str = (rf"Effective Center: {mu_eta:.1f} $\pm$ {mu_eta_err:.1f} $\mu$m" + "\n" + 
                       rf"CS Width ({100*(1-threshold):.0f}-{100*threshold:.0f}%): {cs_width:.1f} $\pm$ {cs_width_error:.1f} $\mu$m")
            ax2.text(0.02, 0.85, eta_str, transform=ax2.transAxes, fontsize=11,
                     bbox=dict(facecolor="white", alpha=0.9, edgecolor="red"))

        ax2.set_xlabel(r"Projected V (Height) [$\mu$m]")
        ax2.set_ylabel(r"Charge Sharing Ratio ($\eta$)")
        ax2.set_ylim(-0.05, 1.05)
        ax2.grid(True, alpha=0.5)
        ax2.legend(loc="lower right")

        plt.tight_layout()
        plt.show()

    return cs_width, cs_width_error, cs_asymmetry, cs_interpad_lines

In [ ]:
cs_width, cs_width_error, cs_asymmetry, cs_interpad_lines = analyze_cs(
    df, voltage, angle_deg, threshold=0.9, make_plot=True
)

In [ ]:
def get_side_robust(x, y, line):
    if abs(line["k"]) > 1.0:
        return x - (y - line["b"]) / line["k"]
    else:
        return y - (line["k"] * x + line["b"])

def calculate_baseline_gap(y_arr, jitter_arr, prominence_percent=15.0):
    if len(y_arr) < 15:
        return (np.nan,) * 6

    rise_fraction = prominence_percent / 100.0

    trim = max(3, int(len(jitter_arr) * 0.1))
    inner_jitter = jitter_arr[trim:-trim]
    if len(inner_jitter) == 0:
        return (np.nan,) * 6
    
    peak_idx = np.argmax(inner_jitter) + trim
    peak_val = jitter_arr[peak_idx]

    left_jitter = jitter_arr[:peak_idx]
    left_y = y_arr[:peak_idx]
    
    right_jitter = jitter_arr[peak_idx+1:]
    right_y = y_arr[peak_idx+1:]

    def get_robust_baseline(j_arr):
        if len(j_arr) < 5:
            return np.nan, np.nan
        sorted_j = np.sort(j_arr)
        p10 = max(0, int(len(sorted_j) * 0.10))
        p50 = max(1, int(len(sorted_j) * 0.50))
        base_region = sorted_j[p10:p50]
        
        return np.mean(base_region), np.std(base_region)

    l_base, l_err = get_robust_baseline(left_jitter)
    r_base, r_err = get_robust_baseline(right_jitter)

    l_prominence = peak_val - l_base
    r_prominence = peak_val - r_base
    
    l_thresh = l_base + rise_fraction * l_prominence if not np.isnan(l_base) else np.nan
    r_thresh = r_base + rise_fraction * r_prominence if not np.isnan(r_base) else np.nan

    x_left, k_left = np.nan, np.nan
    if not np.isnan(l_thresh):
        for i in range(peak_idx, 0, -1):
            if jitter_arr[i] >= l_thresh and jitter_arr[i-1] < l_thresh:
                k_left = (jitter_arr[i] - jitter_arr[i-1]) / (y_arr[i] - y_arr[i-1])
                x_left = y_arr[i-1] + (l_thresh - jitter_arr[i-1]) / k_left
                break
                
    x_right, k_right = np.nan, np.nan
    if not np.isnan(r_thresh):
        for i in range(peak_idx, len(jitter_arr) - 1):
            if jitter_arr[i] >= r_thresh and jitter_arr[i+1] < r_thresh:
                k_right = (jitter_arr[i+1] - jitter_arr[i]) / (y_arr[i+1] - y_arr[i])
                x_right = y_arr[i] + (r_thresh - jitter_arr[i]) / k_right
                break
                
    width, width_err = np.nan, np.nan
    if not np.isnan(x_left) and not np.isnan(x_right):
        width = x_right - x_left
        err_xl = l_err / abs(k_left) if k_left != 0 else 0
        err_xr = r_err / abs(k_right) if k_right != 0 else 0
        width_err = np.sqrt(err_xl**2 + err_xr**2)
        
    return l_base, r_base, x_left, x_right, width, width_err


def plot_jitter_profile(df, percentage, borders, cce_interpad_lines, cs_interpad_lines=None, fit_options=None, make_plot=True, prominence_percent=15.0):
    if fit_options is None:
        fit_options = {
            "bins": 100,
            "left_lim": 98.0,
            "right_lim": 100.0,
            "peak_left": 98.0,
            "peak_right": 100.0,
        }

    df = df.copy()
    for col in ["x", "y"]:
        if col not in df.columns:
            if col in df.index.names or df.index.name == col:
                df = df.reset_index()
            else:
                df[col] = 0.0

    bins = fit_options["bins"]
    p_left, p_right = fit_options["peak_left"], fit_options["peak_right"]

    borders = copy.deepcopy(borders)
    cce_interpad_lines = copy.deepcopy(cce_interpad_lines)
    if cs_interpad_lines is not None:
        cs_interpad_lines = copy.deepcopy(cs_interpad_lines)

    data_x_mid = df["x"].mean()
    current_border_x = (-1700 - borders["left"]["b"]) / borders["left"]["k"]
    if abs(data_x_mid - current_border_x) > 500:
        x_offset = data_x_mid - current_border_x
        borders["left"]["b"] -= x_offset * borders["left"]["k"]
        borders["right"]["b"] -= x_offset * borders["right"]["k"]

    side_bot = get_side_robust(df["x"], df["y"], borders["bottom"])
    side_top = get_side_robust(df["x"], df["y"], borders["top"])
    side_left = get_side_robust(df["x"], df["y"], borders["left"])
    side_right = get_side_robust(df["x"], df["y"], borders["right"])

    mask_full = (side_left * side_right <= 0) & (side_top * side_bot <= 0)
    df_filt = df[mask_full].copy()

    if df_filt.empty:
        return pd.DataFrame(), np.nan, np.nan, np.nan, {}

    channels = sorted(df_filt["ch"].unique())
    ch_y_centers = {}
    
    for ch in channels:
        df_ch = df_filt[df_filt["ch"] == ch]
        if "peak_amplitude" in df_ch.columns:
            weights = np.abs(df_ch["peak_amplitude"])
            if weights.sum() > 0:
                ch_y_centers[ch] = (df_ch["y"] * weights).sum() / weights.sum()
            else:
                ch_y_centers[ch] = df_ch["y"].mean()
        else:
            valid_y = df_ch.dropna(subset=[f"peak_time_{percentage}"])["y"]
            ch_y_centers[ch] = valid_y.mean() if not valid_y.empty else df_ch["y"].mean()

    if not ch_y_centers: 
        return pd.DataFrame(), np.nan, np.nan, np.nan, {}
        
    bottom_ch = min(ch_y_centers, key=ch_y_centers.get)
    top_ch = max(ch_y_centers, key=ch_y_centers.get)

    val_col = f"peak_time_{percentage}"
    charge_col = "peak_amplitude"
    index_cols = [c for c in ["x", "y", "z", "waveform_index", "ch"] if c in df_filt.columns]

    if charge_col in df_filt.columns:
        pivot_pulses = df_filt.pivot_table(index=index_cols, columns="pulse_number", values=[val_col, charge_col]).dropna()
        if pivot_pulses.empty:
            return pd.DataFrame(), np.nan, np.nan, np.nan, {}
        dt_series = pivot_pulses[val_col][2] - pivot_pulses[val_col][1]
        q_series = np.abs(pivot_pulses[charge_col][2])
        calc_df = pd.DataFrame({"dt": dt_series, "q": q_series}).reset_index()
    else:
        pivot_pulses = df_filt.pivot_table(index=index_cols, columns="pulse_number", values=val_col).dropna()
        if pivot_pulses.empty:
            return pd.DataFrame(), np.nan, np.nan, np.nan, {}
        dt_series = pivot_pulses[2] - pivot_pulses[1]
        calc_df = pd.DataFrame({"dt": dt_series, "q": 1.0}).reset_index()

    ip_bot = get_side_robust(calc_df["x"], calc_df["y"], cce_interpad_lines["inner_bottom"])
    ip_top = get_side_robust(calc_df["x"], calc_df["y"], cce_interpad_lines["inner_top"])
    glob_bot = get_side_robust(calc_df["x"], calc_df["y"], borders["bottom"])
    glob_top = get_side_robust(calc_df["x"], calc_df["y"], borders["top"])

    in_bottom_pad = (glob_bot * ip_bot <= 0)
    in_top_pad = (ip_top * glob_top <= 0)
    in_interpad = (ip_bot * ip_top <= 0)

    valid_dt_list = []

    df_bot_pad = calc_df[in_bottom_pad & (calc_df["ch"] == bottom_ch)]
    if not df_bot_pad.empty:
        valid_dt_list.append(df_bot_pad[["x", "y", "dt"]])

    df_top_pad = calc_df[in_top_pad & (calc_df["ch"] == top_ch)]
    if not df_top_pad.empty:
        valid_dt_list.append(df_top_pad[["x", "y", "dt"]])

    df_ip = calc_df[in_interpad]
    if not df_ip.empty:
        event_cols = [c for c in index_cols if c != "ch"]
        pivot_ip = df_ip.pivot_table(index=event_cols, columns="ch", values=["dt", "q"])
        
        if not pivot_ip.empty and bottom_ch in pivot_ip["dt"].columns and top_ch in pivot_ip["dt"].columns:
            dt_b = pivot_ip["dt"][bottom_ch]
            dt_t = pivot_ip["dt"][top_ch]
            q_b = pivot_ip["q"][bottom_ch].fillna(0.0)
            q_t = pivot_ip["q"][top_ch].fillna(0.0)

            dt_weighted = dt_t.where((q_t > q_b) | dt_b.isna(), dt_b)
            df_ip_weighted = pd.DataFrame({"dt": dt_weighted}).dropna().reset_index()
            valid_dt_list.append(df_ip_weighted[["x", "y", "dt"]])
        else:
            fallback_df = df_ip.dropna(subset=["dt"])
            valid_dt_list.append(fallback_df[["x", "y", "dt"]])

    if not valid_dt_list:
        return pd.DataFrame(), np.nan, np.nan, np.nan, {}

    valid_dt_df = pd.concat(valid_dt_list, ignore_index=True)

    y_vals = sorted(valid_dt_df["y"].unique())
    y_plot, sigmas, sigma_errs = [], [], []

    for y_val in y_vals:
        slice_dt = valid_dt_df[valid_dt_df["y"] == y_val]["dt"].values
        if len(slice_dt) < 10: 
            continue

        y_hist, b_hist = np.histogram(slice_dt, bins=bins, range=(p_left, p_right))
        x_hist = (b_hist[:-1] + b_hist[1:]) / 2

        try:
            popt, pcov = curve_fit(
                gauss, x_hist, y_hist, 
                p0=[np.max(y_hist), np.median(slice_dt), 0.05],
                bounds=([0, p_left, 0], [np.inf, p_right, 0.5]) 
            )
            y_plot.append(y_val)
            
            sigmas.append((abs(popt[2]) * 1e3) / np.sqrt(2))
            sigma_errs.append((np.sqrt(np.diag(pcov))[2] * 1e3) / np.sqrt(2))
        except: 
            pass

    profile_df = pd.DataFrame({"y": y_plot, "jitter_ps": sigmas, "jitter_err_ps": sigma_errs})

    jitter_asym_pct = np.nan
    jitter_width, jitter_width_error = np.nan, np.nan
    jitter_interpad_lines = {}

    if not profile_df.empty:
        y_coords = profile_df["y"].values
        l_base, r_base, x_l, x_r, jitter_width, jitter_width_error = calculate_baseline_gap(
            y_coords, profile_df["jitter_ps"].values, prominence_percent=prominence_percent
        )
        
        if not np.isnan(l_base) and not np.isnan(r_base):
            min_base = min(l_base, r_base)
            jitter_asym_pct = (abs(r_base - l_base) / min_base) * 100.0

        if not np.isnan(x_l) and not np.isnan(x_r):
            y_bot = min(x_l, x_r)
            y_top = max(x_l, x_r)
            jitter_interpad_lines = {
                'inner_bottom': {'k': np.float64(0.0), 'b': np.float64(y_bot)},
                'inner_top': {'k': np.float64(0.0), 'b': np.float64(y_top)}
            }

    if make_plot and not profile_df.empty:
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 10), sharex=True)

        x_mid_coord = valid_dt_df["x"].mean()
        
        cce_y_approx = [
            cce_interpad_lines["inner_bottom"]["k"] * x_mid_coord + cce_interpad_lines["inner_bottom"]["b"],
            cce_interpad_lines["inner_top"]["k"] * x_mid_coord + cce_interpad_lines["inner_top"]["b"],
        ]
        
        cs_y_approx = []
        if cs_interpad_lines is not None:
            cs_y_approx = [
                cs_interpad_lines["inner_bottom"]["k"] * x_mid_coord + cs_interpad_lines["inner_bottom"]["b"],
                cs_interpad_lines["inner_top"]["k"] * x_mid_coord + cs_interpad_lines["inner_top"]["b"],
            ]

        y_min_p, y_max_p = valid_dt_df["y"].min(), valid_dt_df["y"].max()
        
        if len(y_vals) > 1:
            step = np.median(np.diff(y_vals))
            y_bins_array = np.arange(y_min_p - step, y_max_p + 2*step, step)
        else:
            y_bins_array = 50 

        h = ax1.hist2d(
            valid_dt_df["y"], valid_dt_df["dt"], 
            bins=[y_bins_array, bins],
            range=[[y_min_p, y_max_p], [p_left, p_right]], 
            cmap="viridis", cmin=1
        )
        
        divider1 = make_axes_locatable(ax1)
        cax1 = divider1.append_axes("right", size="5%", pad=0.1)
        fig.colorbar(h[3], cax=cax1, label="Counts")

        ax1.axvspan(min(cce_y_approx), max(cce_y_approx), color="red", alpha=0.1, label="CCE Gap")
        if cs_y_approx:
            ax1.axvspan(min(cs_y_approx), max(cs_y_approx), color="blue", alpha=0.1, label="CS Region")
            
        ax1.set_ylabel(r"$\Delta t$ [ns]")
        ax1.set_title(f"Jitter Analysis Profile - {percentage}% Threshold")
        ax1.grid(True, alpha=0.3)
        ax1.legend(loc="upper right")

        ax2.errorbar(
            profile_df["y"], profile_df["jitter_ps"], yerr=profile_df["jitter_err_ps"],
            marker="o", linestyle="-", color="indigo", capsize=3, markersize=5, label=r"Fitted Jitter ($\sigma$)"
        )

        if not np.isnan(jitter_width):
            peak_val_plot = profile_df["jitter_ps"].max()
            rose_fraction_plot = prominence_percent / 100.0
            
            l_thresh_plot = l_base + rose_fraction_plot * (peak_val_plot - l_base)
            r_thresh_plot = r_base + rose_fraction_plot * (peak_val_plot - r_base)

            ax2.plot([y_coords[0], x_l], [l_base, l_base], color="green", linestyle="--", linewidth=2.5, label="Robust Baselines")
            ax2.plot([x_r, y_coords[-1]], [r_base, r_base], color="green", linestyle="--", linewidth=2.5)
            
            ax2.plot([x_l-15, x_l], [l_thresh_plot, l_thresh_plot], color="darkorange", linestyle="-", linewidth=2)
            ax2.plot([x_r, x_r+15], [r_thresh_plot, r_thresh_plot], color="darkorange", linestyle="-", linewidth=2)
            
            ax2.axvline(x_l, color="darkorange", linestyle=":", linewidth=2)
            ax2.axvline(x_r, color="darkorange", linestyle=":", linewidth=2)
            
            y_max_jitter = profile_df["jitter_ps"].max()
            y_annot = max(l_base, r_base) + (y_max_jitter - max(l_base, r_base)) * 0.2
            ax2.annotate("", xy=(x_l, y_annot), xytext=(x_r, y_annot), arrowprops=dict(arrowstyle="<->", color="darkorange", lw=2))
            ax2.text((x_l + x_r)/2, y_annot + 0.5, rf"Width: {jitter_width:.1f} $\pm$ {jitter_width_error:.1f} $\mu$m", 
                     ha="center", va="bottom", color="darkorange", fontweight="bold", 
                     bbox=dict(facecolor="white", alpha=0.8, edgecolor="none"))

            asym_str = (f"Left Baseline: {l_base:.1f} ps\n"
                        f"Right Baseline: {r_base:.1f} ps\n"
                        f"Asymmetry: {jitter_asym_pct:+.1f}%\n"
                        f"Prominence Threshold: {prominence_percent}%")
            ax2.text(0.02, 0.82, asym_str, transform=ax2.transAxes, fontsize=11,
                     bbox=dict(facecolor="white", alpha=0.8, edgecolor="green"))

        ax2.axvspan(min(cce_y_approx), max(cce_y_approx), color="red", alpha=0.1, label="CCE Gap")
        if cs_y_approx:
            ax2.axvspan(min(cs_y_approx), max(cs_y_approx), color="blue", alpha=0.1, label="CS Region")
            
        ax2.set_xlabel(r"y-coordinate [$\mu$m]")
        ax2.set_ylabel(r"Jitter ($\sigma$) [ps]")
        ax2.grid(True, alpha=0.3)
        ax2.legend(loc="upper right")

        x_min_lim, x_max_lim = ax2.get_xlim()
        ax2.set_xticks(np.arange(np.ceil(x_min_lim / 20) * 20, x_max_lim, 20))
        
        divider2 = make_axes_locatable(ax2)
        cax2 = divider2.append_axes("right", size="5%", pad=0.1)
        cax2.set_visible(False)

        plt.tight_layout()
        plt.show()

    try:
        profile_df.attrs['baseline_width'] = jitter_width
        profile_df.attrs['baseline_width_err'] = jitter_width_error
        profile_df.attrs['jitter_asymmetry_pct'] = jitter_asym_pct
    except:
        pass

    return profile_df, jitter_width, jitter_width_error, jitter_asym_pct, jitter_interpad_lines

In [ ]:
percentage = 20
profile_df, jitter_width, jitter_width_error, jitter_asymmetry, jitter_interpad_lines = plot_jitter_profile(
    df,
    percentage=percentage,
    borders=borders,
    cce_interpad_lines=cce_interpad_lines,
    # cs_interpad_lines=cs_interpad_lines,
    fit_options={
        "bins": 100,
        "left_lim": 98,
        "right_lim": 99.5,
        "peak_left": 98,
        "peak_right": 99.5,
    },
    make_plot=True,
    prominence_percent=15.0
)

In [ ]:
def plot_jitter(
    df, percentage, borders, interpad_lines, fit_options=None, make_plot=False
):
    if fit_options is None:
        fit_options = {
            "bins": 100,
            "left_lim": 98.0,
            "right_lim": 100.0,
            "peak_left": 98.0,
            "peak_right": 100.0,
        }

    bins = fit_options["bins"]
    l_lim, r_lim = fit_options["left_lim"], fit_options["right_lim"]
    p_left, p_right = fit_options["peak_left"], fit_options["peak_right"]

    borders = copy.deepcopy(borders)
    interpad_lines = copy.deepcopy(interpad_lines)

    data_x_mid = df["x"].mean()
    current_border_x = (-1700 - borders["left"]["b"]) / borders["left"]["k"]

    if abs(data_x_mid - current_border_x) > 500:
        print("WARNING: Aligning borders to data coordinate system.")
        x_offset = data_x_mid - current_border_x
        borders["left"]["b"] -= x_offset * borders["left"]["k"]
        borders["right"]["b"] -= x_offset * borders["right"]["k"]

    side_bot = get_side_robust(df["x"], df["y"], borders["bottom"])
    side_top = get_side_robust(df["x"], df["y"], borders["top"])
    side_left = get_side_robust(df["x"], df["y"], borders["left"])
    side_right = get_side_robust(df["x"], df["y"], borders["right"])

    mask_full = (side_left * side_right <= 0) & (side_top * side_bot <= 0)
    df_filt = df[mask_full].copy()

    if df_filt.empty:
        print("DEBUG: No points passed the Global Mask.")
        return {}, {}

    jitters, jitter_errors = {}, {}
    possible_indices = ["x", "y", "z", "waveform_index", "ch"]
    index_cols = [c for c in possible_indices if c in df.columns]

    if make_plot:
        plt.figure(figsize=(10, 6))

    channels = sorted(df_filt["ch"].unique())

    ch_y_centers = {}
    for ch in channels:
        df_ch = df_filt[df_filt["ch"] == ch]
        if "peak_amplitude" in df_ch.columns:
            weights = np.abs(df_ch["peak_amplitude"])
            if weights.sum() > 0:
                ch_y_centers[ch] = (df_ch["y"] * weights).sum() / weights.sum()
            else:
                ch_y_centers[ch] = df_ch["y"].mean()
        else:
            valid_y = df_ch.dropna(subset=[f"peak_time_{percentage}"])["y"]
            ch_y_centers[ch] = (
                valid_y.mean() if not valid_y.empty else df_ch["y"].mean()
            )

    if not ch_y_centers:
        print("DEBUG: No valid channels found.")
        return {}, {}

    bottom_ch = min(ch_y_centers, key=ch_y_centers.get)

    all_valid_jitters = []

    for ch in channels:
        df_ch = df_filt[df_filt["ch"] == ch].copy()
        side_ip_bot = get_side_robust(
            df_ch["x"], df_ch["y"], interpad_lines["inner_bottom"]
        )
        side_ip_top = get_side_robust(
            df_ch["x"], df_ch["y"], interpad_lines["inner_top"]
        )

        if ch == bottom_ch:
            mask_pad = (
                get_side_robust(df_ch["x"], df_ch["y"], borders["bottom"]) * side_ip_bot
                <= 0
            )
            label_base = f"Ch {ch} (Bottom Pad)"
        else:
            mask_pad = (
                side_ip_top * get_side_robust(df_ch["x"], df_ch["y"], borders["top"])
                <= 0
            )
            label_base = f"Ch {ch} (Top Pad)"

        df_final = df_ch[mask_pad]
        if df_final.empty:
            continue

        val_col = f"peak_time_{percentage}"
        pivot = df_final.pivot_table(
            index=index_cols, columns="pulse_number", values=val_col
        ).dropna()

        if not pivot.empty:
            jitter_data = (pivot[2] - pivot[1]).values
            all_valid_jitters.append(jitter_data)

            y, b = np.histogram(jitter_data, bins=bins, range=(p_left, p_right))
            x_vals = (b[:-1] + b[1:]) / 2
            try:
                popt, pcov = curve_fit(
                    gauss, x_vals, y, p0=[np.max(y), x_vals[np.argmax(y)], 0.05]
                )
                
                sigma = abs(popt[2]) / np.sqrt(2)
                jitters[f"ch_{ch}"] = sigma
                jitter_errors[f"ch_{ch}"] = np.sqrt(np.diag(pcov))[2] / np.sqrt(2)

                if make_plot:
                    _, _, patches = plt.hist(
                        jitter_data,
                        bins=bins,
                        range=(l_lim, r_lim),
                        alpha=0.6,
                        label=rf"{label_base}: $\sigma$={1e3*sigma:.1f} ps",
                        zorder=3,
                    )
                    hist_color = patches[0].get_facecolor()[:3]

                    x_fit = np.linspace(p_left, p_right, 200)
                    plt.plot(
                        x_fit, gauss(x_fit, *popt), lw=2, color=hist_color, zorder=4
                    )
            except (RuntimeError, ValueError):
                pass

    side_ip_bot_all = get_side_robust(
        df_filt["x"], df_filt["y"], interpad_lines["inner_bottom"]
    )
    side_ip_top_all = get_side_robust(
        df_filt["x"], df_filt["y"], interpad_lines["inner_top"]
    )
    df_ip = df_filt[side_ip_bot_all * side_ip_top_all <= 0]

    if not df_ip.empty:
        val_col = f"peak_time_{percentage}"
        charge_col = "peak_amplitude" 
        
        jitter_ip = np.array([])
        
        if charge_col in df_ip.columns:
            pivot_pulses = df_ip.pivot_table(
                index=index_cols, 
                columns="pulse_number", 
                values=[val_col, charge_col]
            ).dropna()
            
            if not pivot_pulses.empty:
                dt = pivot_pulses[val_col][2] - pivot_pulses[val_col][1]
                q = np.abs(pivot_pulses[charge_col][2])
                
                df_calc = pd.DataFrame({"dt": dt, "q": q}).reset_index()
                event_cols = [c for c in index_cols if c != "ch"]
                
                pivot_ch = df_calc.pivot_table(
                    index=event_cols, 
                    columns="ch", 
                    values=["dt", "q"]
                ).dropna()
                
                if not pivot_ch.empty and len(pivot_ch["dt"].columns) >= 2:
                    ch_keys = list(pivot_ch["dt"].columns)
                    chA, chB = ch_keys[0], ch_keys[1]
                    
                    dt_A = pivot_ch["dt"][chA]
                    dt_B = pivot_ch["dt"][chB]
                    q_A = pivot_ch["q"][chA]
                    q_B = pivot_ch["q"][chB]
                    
                    jitter_ip = ((dt_A * q_A + dt_B * q_B) / (q_A + q_B)).values
                elif not pivot_ch.empty:
                    jitter_ip = dt.values
        else:
            pivot_ip = df_ip.pivot_table(
                index=index_cols, columns="pulse_number", values=val_col
            ).dropna()
            if not pivot_ip.empty:
                jitter_ip = (pivot_ip[2] - pivot_ip[1]).values

        if len(jitter_ip) > 0:
            all_valid_jitters.append(jitter_ip)

            y_i, b_i = np.histogram(jitter_ip, bins=bins, range=(p_left, p_right))
            x_vals_i = (b_i[:-1] + b_i[1:]) / 2
            try:
                popt_i, pcov_i = curve_fit(
                    gauss, x_vals_i, y_i, p0=[np.max(y_i), np.median(jitter_ip), 0.05]
                )
                
                sigma_i = abs(popt_i[2]) / np.sqrt(2)
                jitters["interpad"] = sigma_i
                jitter_errors["interpad"] = np.sqrt(np.diag(pcov_i))[2] / np.sqrt(2)

                if make_plot:
                    _, _, patches = plt.hist(
                        jitter_ip,
                        bins=bins,
                        range=(l_lim, r_lim),
                        alpha=0.6,
                        label=rf"Inter-pad: $\sigma$={1e3*sigma_i:.1f} ps",
                        zorder=5,
                    )
                    hist_color = patches[0].get_facecolor()[:3]

                    x_fit_i = np.linspace(p_left, p_right, 200)
                    plt.plot(
                        x_fit_i,
                        gauss(x_fit_i, *popt_i),
                        lw=2,
                        color=hist_color,
                        zorder=6,
                    )
            except (RuntimeError, ValueError):
                pass

    if all_valid_jitters:
        jitter_all = np.concatenate(all_valid_jitters)
        if len(jitter_all) > 0:
            y_all, b_all = np.histogram(jitter_all, bins=bins, range=(p_left, p_right))
            x_vals_all = (b_all[:-1] + b_all[1:]) / 2
            try:
                popt_all, pcov_all = curve_fit(
                    gauss,
                    x_vals_all,
                    y_all,
                    p0=[np.max(y_all), np.median(jitter_all), 0.05],
                )
                
                sigma_all = abs(popt_all[2]) / np.sqrt(2)
                jitters["full"] = sigma_all
                jitter_errors["full"] = np.sqrt(np.diag(pcov_all))[2] / np.sqrt(2)

                if make_plot:
                    _, _, patches = plt.hist(
                        jitter_all,
                        bins=bins,
                        range=(l_lim, r_lim),
                        alpha=0.2,
                        label=rf"Full Region: $\sigma$={1e3*sigma_all:.1f} ps",
                        zorder=1,
                    )
                    hist_color = patches[0].get_facecolor()[:3]

                    x_fit_all = np.linspace(p_left, p_right, 200)
                    plt.plot(
                        x_fit_all,
                        gauss(x_fit_all, *popt_all),
                        color=hist_color,
                        lw=2,
                        zorder=2,
                    )
            except (RuntimeError, ValueError):
                pass

    if make_plot:
        plt.title(f"Jitter Analysis - {percentage}% Threshold")
        plt.xlabel(r"$\Delta t$ [ns]")
        plt.ylabel("Counts")
        plt.xlim(l_lim, r_lim)
        plt.legend(loc="upper right")
        plt.grid(True, alpha=0.3, zorder=0)
        plt.show()

    return jitters, jitter_errors

In [ ]:
percentage = 20
jitters, jitter_errors = plot_jitter(
    df,
    percentage,
    borders=borders,
    interpad_lines=jitter_interpad_lines,
    fit_options={
        "bins": 100,
        "left_lim": 98,
        "right_lim": 99.5,
        "peak_left": 98,
        "peak_right": 99.5,
    },
    make_plot=True,
)

In [ ]:
def plot_waveform(sample_path, voltage, position, channel, index=0):
    target_pos = np.array(position)

    with h5py.File(sample_path, "r") as f:
        grp_name = f"voltage_{voltage:04d}V"
        if grp_name not in f:
            print(f"Error: Group {grp_name} not found in HDF5 file.")
            return

        voltage_grp = f[grp_name]
        closest_key = None
        closest_pos = None
        min_dist = float("inf")

        for key in voltage_grp.keys():
            subgroup = voltage_grp[key]

            if all(k in subgroup.attrs for k in ("x", "y", "z")):
                curr_x = 1e6 * subgroup.attrs["x"]
                curr_y = 1e6 * subgroup.attrs["y"]
                curr_z = 1e6 * subgroup.attrs["z"]

                current_pos = np.array([curr_x, curr_y, curr_z])

                dist = np.linalg.norm(current_pos - target_pos)

                if dist < min_dist:
                    min_dist = dist
                    closest_key = key
                    closest_pos = current_pos

        if closest_key is None:
            print("Could not find any groups with x, y, z attributes.")
            return

        if min_dist == 0:
            print(f"Found exact match: {closest_pos} in group '{closest_key}'")
        else:
            print(f"Position {position} not found.")
            print(
                f"Closest match: {closest_pos} in group '{closest_key}' (Dist: {min_dist:.2f})"
            )

        pos_grp = voltage_grp[closest_key]

        ds_name = f"ch{channel}_v"

        ch_data = pos_grp[ds_name]

        dt = ch_data.attrs["dt"]
        t0 = ch_data.attrs["t0"]

        n_samples = ch_data.shape[1]
        half = n_samples // 2

        t = t0 + np.arange(n_samples) * dt
        t *= 1e9

        waveforms = ch_data[:][index]
        plt.plot(t[:half], waveforms[:half], label="Pulse 1")
        plt.plot(t[half:], waveforms[half:], label="Pulse 2")
        plt.xlabel("t, [ns]")
        plt.ylabel("Amplitude, [V]")
        plt.title(
            f"{voltage} V, ch={channel} x={position[0]} y={position[1]} z={position[2]}, wf_idx={index}"
        )
        plt.legend()

In [ ]:
position, ch = [-2440, -350, 68000], 1

plot_waveform(
    f"{data_folder}/{wafer_type}/{sample_name}/{sample_name}.h5",
    voltage,
    position,
    ch,
    76,
)


In [ ]:
def calc_spatial_mean(agg_df, mask, col_name="total_collected_charge"):
    data = np.abs(agg_df[mask][col_name])
    if len(data) == 0:
        return np.nan, np.nan
    return data.mean(), data.std() / np.sqrt(len(data))

def analyze_collected_charge_map(
    df, voltage, borders=None, interpad_lines=None, make_plot=False
):
    mean_per_ch = (
        df[df["pulse_number"] == 1]
        .groupby(["x", "y", "ch"], as_index=False)["peak_integral"]
        .mean()
    )

    channels = sorted(mean_per_ch["ch"].unique())
    ch_y_centers = {}

    for ch in channels:
        df_ch = mean_per_ch[mean_per_ch["ch"] == ch]
        weights = np.abs(df_ch["peak_integral"])
        if weights.sum() > 0:
            weighted_y = (df_ch["y"] * weights).sum() / weights.sum()
        else:
            weighted_y = df_ch["y"].mean()
        ch_y_centers[ch] = weighted_y

    if ch_y_centers:
        bottom_ch = min(ch_y_centers, key=ch_y_centers.get)
        top_ch = max(ch_y_centers, key=ch_y_centers.get)
    else:
        bottom_ch, top_ch = None, None

    pivot_ch = mean_per_ch.pivot(index=["x", "y"], columns="ch", values="peak_integral").fillna(0).reset_index()
    ch_cols = [c for c in pivot_ch.columns if c not in ["x", "y"]]
    
    agg_df = pivot_ch[["x", "y"]].copy()
    abs_ch_data = np.abs(pivot_ch[ch_cols])

    mean_charges = {}
    charge_errors = {}

    if borders is not None and interpad_lines is not None:
        side_bot = get_side_robust(agg_df["x"], agg_df["y"], borders["bottom"])
        side_top = get_side_robust(agg_df["x"], agg_df["y"], borders["top"])
        side_left = get_side_robust(agg_df["x"], agg_df["y"], borders["left"])
        side_right = get_side_robust(agg_df["x"], agg_df["y"], borders["right"])

        mask_full = (side_left * side_right <= 0) & (side_top * side_bot <= 0)

        side_ip_bot = get_side_robust(
            agg_df["x"], agg_df["y"], interpad_lines["inner_bottom"]
        )
        side_ip_top = get_side_robust(
            agg_df["x"], agg_df["y"], interpad_lines["inner_top"]
        )

        mask_ip = (side_ip_bot * side_ip_top <= 0)
        
        agg_df["total_collected_charge"] = np.where(
            mask_ip,
            abs_ch_data.sum(axis=1),
            abs_ch_data.max(axis=1)
        )

        if mask_full.any():
            mean, err = calc_spatial_mean(agg_df, mask_full)
            mean_charges["full"] = mean
            charge_errors["full"] = err

        mask_ip_full = mask_full & mask_ip
        if mask_ip_full.any():
            mean, err = calc_spatial_mean(agg_df, mask_ip_full)
            mean_charges["interpad"] = mean
            charge_errors["interpad"] = err

        mask_top_pad = mask_full & (side_ip_top * side_top <= 0)
        if mask_top_pad.any() and top_ch is not None:
            mean, err = calc_spatial_mean(agg_df, mask_top_pad)
            mean_charges[f"ch_{top_ch}"] = mean
            charge_errors[f"ch_{top_ch}"] = err

        mask_bot_pad = mask_full & (side_ip_bot * side_bot <= 0)
        if mask_bot_pad.any() and bottom_ch is not None:
            mean, err = calc_spatial_mean(agg_df, mask_bot_pad)
            mean_charges[f"ch_{bottom_ch}"] = mean
            charge_errors[f"ch_{bottom_ch}"] = err

    else:
        agg_df["total_collected_charge"] = abs_ch_data.max(axis=1)

    if make_plot:
        pivot = agg_df.pivot(index="y", columns="x", values="total_collected_charge")
        pivot = pivot.sort_index().sort_index(axis=1)

        x_vals = pivot.columns.values
        y_vals = pivot.index.values

        dx = np.abs(np.median(np.diff(x_vals)))
        dy = np.abs(np.median(np.diff(y_vals)))

        _, ax = plt.subplots(figsize=(10, 10))

        sns.heatmap(
            np.abs(pivot),
            cmap="plasma",
            cbar_kws={"label": r"Total Collected Charge [V$\cdot$ns]"},
            ax=ax,
        )

        if pd.notna(dx) and pd.notna(dy) and dx > 0:
            ax.set_aspect(dy / dx)

        ax.set_title(f"TCT Scan - {voltage} V - Total Collected Charge Map")
        ax.set_xlabel(r"x ($\mu$m)")
        ax.set_ylabel(r"y ($\mu$m)")

        tick_spacing = 5

        ax.set_xticks(np.arange(len(x_vals))[::tick_spacing] + 0.5)
        ax.set_xticklabels(x_vals[::tick_spacing].astype(int), rotation=45)

        ax.set_yticks(np.arange(len(y_vals))[::tick_spacing] + 0.5)
        ax.set_yticklabels(y_vals[::tick_spacing].astype(int), rotation=0)

        if borders is not None:
            corners_raw = [
                get_intersection(borders["left"], borders["bottom"]),
                get_intersection(borders["right"], borders["bottom"]),
                get_intersection(borders["right"], borders["top"]),
                get_intersection(borders["left"], borders["top"]),
            ]

            pixels = [to_plot_coords(x, y, x_vals, y_vals) for x, y in corners_raw]

            rect = Polygon(
                pixels,
                closed=True,
                linewidth=2,
                edgecolor="black",
                facecolor="none",
            )
            ax.add_patch(rect)

            if interpad_lines is not None:
                for _, line_data in interpad_lines.items():
                    p1_raw = get_intersection(borders["left"], line_data)
                    p2_raw = get_intersection(borders["right"], line_data)

                    px1, py1 = to_plot_coords(*p1_raw, x_vals, y_vals)
                    px2, py2 = to_plot_coords(*p2_raw, x_vals, y_vals)

                    ax.plot(
                        [px1, px2],
                        [py1, py2],
                        color="black",
                        linewidth=2,
                        linestyle="-",
                    )

        plt.tight_layout()
        plt.show()

    return mean_charges, charge_errors

In [ ]:
mean_charges, charge_errors = analyze_collected_charge_map(
    df, voltage, borders=borders, interpad_lines=cs_interpad_lines, make_plot=True
)

In [ ]:
# csv_file = f"{data_folder}/{wafer_type}/{sample_name}/analyzed_data.csv"

# active_chs = sorted([k for k in jitters.keys() if k.startswith("ch_")])

# ch_a_key = active_chs[0] if len(active_chs) > 0 else "ch_A_missing"
# ch_b_key = active_chs[1] if len(active_chs) > 1 else "ch_B_missing"

# new_row_df = pd.DataFrame([{
#     "voltage": voltage,
#     "interpad_distance": interpad_distance,
#     "interpad_distance_error": interpad_distance_error,
#     "interpad_jitter": jitters.get("interpad", np.nan),
#     "interpad_jitter_error": jitter_errors.get("interpad", np.nan),
#     "full_jitter": jitters.get("full", np.nan),
#     "full_jitter_error": jitter_errors.get("full", np.nan),

#     "pad_1_name": ch_a_key,
#     "pad_1_jitter": jitters.get(ch_a_key, np.nan),
#     "pad_1_jitter_error": jitter_errors.get(ch_a_key, np.nan),

#     "pad_2_name": ch_b_key,
#     "pad_2_jitter": jitters.get(ch_b_key, np.nan),
#     "pad_2_jitter_error": jitter_errors.get(ch_b_key, np.nan),

#     "pad_1_mean_charge": mean_charges.get(ch_a_key, np.nan),
#     "pad_1_mean_charge_error": charge_errors.get(ch_a_key, np.nan),
#     "pad_2_mean_charge": mean_charges.get(ch_b_key, np.nan),
#     "pad_2_mean_charge_error": charge_errors.get(ch_b_key, np.nan),

#     "interpad_mean_charge": mean_charges.get("interpad", np.nan),
#     "interpad_mean_charge_error": charge_errors.get("interpad", np.nan),
#     "full_mean_charge": mean_charges.get("full", np.nan),
#     "full_mean_charge_error": charge_errors.get("full", np.nan)
# }])

# if os.path.isfile(csv_file):
#     existing_df = pd.read_csv(csv_file)

#     combined_df = pd.concat([existing_df, new_row_df], ignore_index=True)

#     combined_df.drop_duplicates(subset=['voltage'], keep='last', inplace=True)

#     combined_df.sort_values(by='voltage', inplace=True)

#     combined_df.to_csv(csv_file, index=False)
# else:
#     new_row_df.to_csv(csv_file, index=False)

# print(f"Successfully wrote data of {voltage}V to {csv_file}")

In [ ]:
voltages = [50, 60, 70, 80, 90, 100, 110, 120, 130, 140, 150]
wafer_type = "W2"
sample_names = ["TW5_V1"]#, "TW5_V2", "TW1_V2", "TW2_V2"]
percentage = 20

for sample_name in sample_names:
    for voltage in voltages:
        file_path = f"{data_folder}/{wafer_type}/{sample_name}/{sample_name}_{voltage}V.csv"
        
        if not os.path.exists(file_path):
            print(f"Skipping {voltage}V: File not found.")
            continue
            
        print(f"Processing {sample_name} at {voltage}V...")
        df = read_analysis(file_path)

        borders, angle_deg = get_sensor_borders(df, threshold_fraction=0.7)

        cce_interpad_lines, cce_interpad_distance, cce_interpad_distance_error = analyze_cce(
            df, voltage, angle_deg=angle_deg, threshold=0.9, make_plot=False, normalization=True
        )


        print(cce_interpad_distance)

        cs_width, cs_width_error, cs_asymmetry, cs_interpad_lines = analyze_cs(
            df, voltage, angle_deg=angle_deg, threshold=0.9, make_plot=False
        )

        profile_df, jitter_width, jitter_width_error, jitter_asymmetry, jitter_interpad_lines = plot_jitter_profile(
            df,
            percentage=percentage,
            borders=borders,
            cce_interpad_lines=cce_interpad_lines,
            cs_interpad_lines=cs_interpad_lines,
            fit_options={
                "bins": 100,
                "left_lim": 98,
                "right_lim": 99.5,
                "peak_left": 98,
                "peak_right": 99.5,
            },
            make_plot=False,
        )

        jitters, jitter_errors = plot_jitter(
            df,
            percentage,
            borders=borders,
            interpad_lines=jitter_interpad_lines,
            fit_options={
                "bins": 100,
                "left_lim": 98,
                "right_lim": 99.5,
                "peak_left": 98,
                "peak_right": 99.5,
            },
            make_plot=False,
        )

        mean_charges, charge_errors = analyze_collected_charge_map(
            df, voltage, borders=borders, interpad_lines=cs_interpad_lines, make_plot=False
        )

        csv_file = f"{data_folder}/{wafer_type}/{sample_name}/analyzed_data.csv"
        os.makedirs(os.path.dirname(csv_file), exist_ok=True)


        active_chs = sorted([k for k in jitters.keys() if k.startswith("ch_")])
        ch_a_key = active_chs[0] if len(active_chs) > 0 else "ch_A_missing"
        ch_b_key = active_chs[1] if len(active_chs) > 1 else "ch_B_missing"

        new_row_df = pd.DataFrame([{
            "voltage": voltage,
            
            "cce_gap_distance": cce_interpad_distance,
            "cce_gap_distance_error": cce_interpad_distance_error,
            
            "cs_width": cs_width,
            "cs_width_error": cs_width_error,
            "cs_asymmetry": cs_asymmetry,
            
            "jitter_width": jitter_width,
            "jitter_width_error": jitter_width_error,
            "jitter_asymmetry": jitter_asymmetry,

            "interpad_jitter": jitters.get("interpad", np.nan),
            "interpad_jitter_error": jitter_errors.get("interpad", np.nan),
            "full_jitter": jitters.get("full", np.nan),
            "full_jitter_error": jitter_errors.get("full", np.nan),

            "pad_1_name": ch_a_key,
            "pad_1_jitter": jitters.get(ch_a_key, np.nan),
            "pad_1_jitter_error": jitter_errors.get(ch_a_key, np.nan),

            "pad_2_name": ch_b_key,
            "pad_2_jitter": jitters.get(ch_b_key, np.nan),
            "pad_2_jitter_error": jitter_errors.get(ch_b_key, np.nan),

            "pad_1_mean_charge": mean_charges.get(ch_a_key, np.nan),
            "pad_1_mean_charge_error": charge_errors.get(ch_a_key, np.nan),
            "pad_2_mean_charge": mean_charges.get(ch_b_key, np.nan),
            "pad_2_mean_charge_error": charge_errors.get(ch_b_key, np.nan),

            "interpad_mean_charge": mean_charges.get("interpad", np.nan),
            "interpad_mean_charge_error": charge_errors.get("interpad", np.nan),
            "full_mean_charge": mean_charges.get("full", np.nan),
            "full_mean_charge_error": charge_errors.get("full", np.nan)
        }])

        if os.path.isfile(csv_file):
            existing_df = pd.read_csv(csv_file)
            combined_df = pd.concat([existing_df, new_row_df], ignore_index=True)
            combined_df.drop_duplicates(subset=['voltage'], keep='last', inplace=True)
            combined_df.sort_values(by='voltage', inplace=True)
            combined_df.to_csv(csv_file, index=False)
        else:
            new_row_df.to_csv(csv_file, index=False)

print("Batch analysis complete!")

In [ ]:
sample_list = [
    ["W2", "TW5_V1", [70,100]],
    ["W2", "TW5_V2", []],
    ["W2", "TW1_V2", []],
    ["W2", "TW2_V2", []],

    # ["W3", "TW5_V1", []],
    # ["W3", "TW5_V2", []],
    # ["W3", "TW1_V2", []],
    # ["W3", "TW2_V2", []],
    
    # ["W4", "TW5_V1", []],
    # ["W4", "TW5_V2", [50,60,65,70,80,90,100,110,120,130,140,150]],
    # ["W4", "TW1_V2", []],
    # ["W4", "TW2_V2", []],

    # ["W5", "TW5_V1", []],
    # ["W5", "TW5_V2", []],
    # ["W5", "TW1_V2", []],
    # ["W5", "TW2_V2", []],
    
    # ["W7", "TW5_V1", []],
    # ["W7", "TW5_V2", []],
    # ["W7", "TW1_V2", []],
    # ["W7", "TW2_V2", []],

    # ["W9", "TW5_V1", []],
    # ["W9", "TW5_V2", []],
    
    # ["W11", "TW5_V1", []],
    # ["W11", "TW5_V2", []],
    # ["W11", "TW1_V2", []],
    # ["W11", "TW2_V2", []],
]

sample_list = [
    ["W2", "TW5_V1", [100]],
    ["W3", "TW5_V1", []],
    ["W4", "TW5_V1", []],
    ["W5", "TW5_V1", []],
    ["W7", "TW5_V1", []],
    ["W11", "TW5_V1", []],
    ["W9", "TW5_V1", []],

    # ["W2", "TW5_V2", []],
    # ["W3", "TW5_V2", []],
    # ["W4", "TW5_V2", [50,60,65,70,80,90,100,110,120,130,140,150]],
    # ["W5", "TW5_V2", []],
    # ["W7", "TW5_V2", []],
    # ["W11", "TW5_V2", []],
    # ["W9", "TW5_V2", []],
    
    # ["W2", "TW1_V2", []],
    # ["W3", "TW1_V2", []],
    # ["W4", "TW1_V2", []],
    # ["W5", "TW1_V2", []],
    # ["W7", "TW1_V2", []],
    # ["W11", "TW1_V2", []],
    
    # ["W2", "TW2_V2", []],
    # ["W3", "TW2_V2", []],
    # ["W4", "TW2_V2", []],
    # ["W5", "TW2_V2", []],
    # ["W7", "TW2_V2", []],
    # ["W11", "TW2_V2", []],


    # ["W2", "TW5_V1", [100]],
    # ["W11", "TW5_V2",[]],
    # ["W9", "TW5_V1",[]],
    # ["W5", "TW5_V1",[]],
    # ["W11", "TW1_V2",[]],
    # ["W11", "TW5_V1",[]],

    # ["PPS/W8","TW2_V1",[]]
]

In [ ]:
plt.figure(figsize=(10, 6))

for wafer_type, sample_name, exclusions in sample_list:
    file_path = f"{data_folder}/{wafer_type}/{sample_name}/analyzed_data.csv"
    df = pd.read_csv(file_path)

    if exclusions:
        df = df[~df["voltage"].isin(exclusions)]

    voltages = df["voltage"].values
    cs_widths = df["cs_width"].values
    cs_width_errors = df["cs_width_error"].values

    if len(voltages) > 0:
        voltages_sorted, cs_widths_sorted, cs_width_errors_sorted = zip(
            *sorted(zip(voltages, cs_widths, cs_width_errors))
        )

        plt.errorbar(
            voltages_sorted,
            cs_widths_sorted,
            yerr=cs_width_errors_sorted,
            marker="o",
            capsize=4,
            label=f"{wafer_type} {sample_name}",
        )
    else:
        plt.plot([], [], marker="o", label=f"{wafer_type} {sample_name}")

plt.title("Charge Sharing (CS) Width")
plt.xlabel("Bias voltage, [V]")
plt.ylabel(r"CS Width, [$\mu$m]")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# plt.figure(figsize=(10, 6))

# for wafer_type, sample_name, exclusions in sample_list:
#     file_path = f"{data_folder}/{wafer_type}/{sample_name}/analyzed_data.csv"
#     df = pd.read_csv(file_path)

#     if exclusions:
#         df = df[~df["voltage"].isin(exclusions)]

#     voltages = df["voltage"].values
#     cs_asym = df["cs_asymmetry"].values

#     if len(voltages) > 0:
#         voltages_sorted, cs_asym_sorted = zip(
#             *sorted(zip(voltages, cs_asym))
#         )

#         plt.plot(
#             voltages_sorted,
#             cs_asym_sorted,
#             marker="o",
#             label=f"{wafer_type} {sample_name}",
#         )
#     else:
#         plt.plot([], [], marker="o", label=f"{wafer_type} {sample_name}")

# plt.title("Charge Sharing Asymmetry")
# plt.xlabel("Bias voltage, [V]")
# plt.ylabel("CS Asymmetry, [%]")
# plt.grid(True)
# plt.legend()
# plt.tight_layout()
# plt.show()

In [ ]:
# plt.figure(figsize=(10, 6))

# for wafer_type, sample_name, exclusions in sample_list:
#     file_path = f"{data_folder}/{wafer_type}/{sample_name}/analyzed_data.csv"
#     df = pd.read_csv(file_path)

#     if exclusions:
#         df = df[~df["voltage"].isin(exclusions)]

#     voltages = df["voltage"].values
#     jitter_widths = df["jitter_width"].values
#     jitter_width_errors = df["jitter_width_error"].values

#     if len(voltages) > 0:
#         voltages_sorted, jitter_widths_sorted, jitter_width_errors_sorted = zip(
#             *sorted(zip(voltages, jitter_widths, jitter_width_errors))
#         )

#         plt.errorbar(
#             voltages_sorted,
#             jitter_widths_sorted,
#             yerr=jitter_width_errors_sorted,
#             marker="o",
#             capsize=4,
#             label=f"{wafer_type} {sample_name}",
#         )
#     else:
#         plt.plot([], [], marker="o", label=f"{wafer_type} {sample_name}")

# plt.title("Jitter Region Width (Between Baselines)")
# plt.xlabel("Bias voltage, [V]")
# plt.ylabel(r"Jitter Width, [$\mu$m]")
# plt.grid(True)
# plt.legend()
# plt.tight_layout()
# plt.show()

In [ ]:
# plt.figure(figsize=(10, 6))

# for wafer_type, sample_name, exclusions in sample_list:
#     file_path = f"{data_folder}/{wafer_type}/{sample_name}/analyzed_data.csv"
#     df = pd.read_csv(file_path)

#     if exclusions:
#         df = df[~df["voltage"].isin(exclusions)]

#     voltages = df["voltage"].values
#     jitter_asym = df["jitter_asymmetry"].values

#     if len(voltages) > 0:
#         voltages_sorted, jitter_asym_sorted = zip(
#             *sorted(zip(voltages, jitter_asym))
#         )

#         plt.plot(
#             voltages_sorted,
#             jitter_asym_sorted,
#             marker="o",
#             label=f"{wafer_type} {sample_name}",
#         )
#     else:
#         plt.plot([], [], marker="o", label=f"{wafer_type} {sample_name}")

# plt.title("Jitter Baseline Asymmetry")
# plt.xlabel("Bias voltage, [V]")
# plt.ylabel("Jitter Asymmetry, [%]")
# plt.grid(True)
# plt.legend()
# plt.tight_layout()
# plt.show()

In [ ]:
from matplotlib.ticker import MultipleLocator

table_data = []

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

for wafer_type, sample_name, exclusions in sample_list:
    file_path = f"{data_folder}/{wafer_type}/{sample_name}/analyzed_data.csv"
    df = pd.read_csv(file_path)

    if exclusions:
        df = df[~df["voltage"].isin(exclusions)]

    df_150 = df[df["voltage"] == 150]
    if not df_150.empty:
        ch1_val = df_150["pad_1_jitter"].values[0]
        ch2_val = df_150["pad_2_jitter"].values[0]
        mean_val = (ch1_val + ch2_val) / 2.0
        
        table_data.append({
            "Sample": f"{wafer_type} {sample_name}",
            "Mean Ch Jitter (ps)": round(mean_val * 1e3, 2),
            "Interpad Jitter (ps)": round(df_150["interpad_jitter"].values[0] * 1e3, 2)
        })

    voltages = df["voltage"].values
    interpad_jitter = df["interpad_jitter"].values
    interpad_jitter_error = df["interpad_jitter_error"].values
    
    pad_1_jitter = df["pad_1_jitter"].values
    pad_1_jitter_error = df["pad_1_jitter_error"].values
    pad_2_jitter = df["pad_2_jitter"].values
    pad_2_jitter_error = df["pad_2_jitter_error"].values

    mean_ch_jitter = (pad_1_jitter + pad_2_jitter) / 2.0
    mean_ch_jitter_error = np.sqrt(pad_1_jitter_error**2 + pad_2_jitter_error**2) / 2.0

    if len(voltages) > 0:
        voltages_sorted, interpad_jitter_sorted, interpad_jitter_error_sorted = zip(
            *sorted(zip(voltages, 1e3 * interpad_jitter, 1e3 * interpad_jitter_error))
        )
        _, mean_ch_jitter_sorted, mean_ch_jitter_error_sorted = zip(
            *sorted(zip(voltages, 1e3 * mean_ch_jitter, 1e3 * mean_ch_jitter_error))
        )

        line1 = ax1.errorbar(
            voltages_sorted,
            mean_ch_jitter_sorted,
            yerr=mean_ch_jitter_error_sorted,
            marker="o",
            capsize=4,
            label=f"{wafer_type} {sample_name}",
        )
        color = line1[0].get_color()

        ax2.errorbar(
            voltages_sorted,
            interpad_jitter_sorted,
            yerr=interpad_jitter_error_sorted,
            marker="o",
            capsize=4,
            label=f"{wafer_type} {sample_name}",
            color=color,
        )

    else:
        line_empty = ax1.errorbar([], [], yerr=[], marker="o", label=f"{wafer_type} {sample_name} Mean")
        color = line_empty[0].get_color()
        ax2.errorbar([], [], yerr=[], marker="o", label=f"{wafer_type} {sample_name}", color=color)

ax1.set_title("Mean jitter of the channels")
ax1.set_xlabel("Bias voltage, [V]")
ax1.set_ylabel("Jitter, [ps]")
ax1.grid(True)
ax1.legend()
ax1.set_ylim(10, 150)
ax1.yaxis.set_major_locator(MultipleLocator(10)) 

ax2.set_title("Inter-pad")
ax2.set_xlabel("Bias voltage, [V]")
ax2.grid(True)
ax2.legend()
ax2.set_ylim(10, 220)
ax2.yaxis.set_major_locator(MultipleLocator(10)) 

plt.tight_layout()

plt.savefig("plot.pdf", format="pdf", dpi=300, bbox_inches="tight")

print("\n" + "="*65)
print("Jitter values at 150V")
print("="*65)
if table_data:
    df_table = pd.DataFrame(table_data)
    df_table.to_csv("table.csv", index=False)
    print(df_table.to_string(index=False))
else:
    print("No data found at 150V for any samples.")
print("="*65 + "\n")

plt.show()

In [ ]:
# fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 6))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

for wafer_type, sample_name, exclusions in sample_list:
    file_path = f"{data_folder}/{wafer_type}/{sample_name}/analyzed_data.csv"
    df = pd.read_csv(file_path)

    if exclusions:
        df = df[~df["voltage"].isin(exclusions)]

    voltages = df["voltage"].values
    interpad_mean_charge = df["interpad_mean_charge"].values
    interpad_mean_charge_error = df["interpad_mean_charge_error"].values
    
    pad_1_mean_charge = df["pad_1_mean_charge"].values
    pad_1_mean_charge_error = df["pad_1_mean_charge_error"].values
    pad_2_mean_charge = df["pad_2_mean_charge"].values
    pad_2_mean_charge_error = df["pad_2_mean_charge_error"].values

    mean_ch_charge = (pad_1_mean_charge + pad_2_mean_charge) / 2.0
    mean_ch_charge_error = np.sqrt(pad_1_mean_charge_error**2 + pad_2_mean_charge_error**2) / 2.0

    if len(voltages) > 0:
        (
            voltages_sorted,
            interpad_mean_charge_sorted,
            interpad_mean_charge_error_sorted,
        ) = zip(
            *sorted(
                zip(
                    voltages,
                    1e3 * interpad_mean_charge,
                    1e3 * interpad_mean_charge_error,
                )
            )
        )
        
        _, mean_ch_charge_sorted, mean_ch_charge_error_sorted = zip(
            *sorted(
                zip(voltages, 1e3 * mean_ch_charge, 1e3 * mean_ch_charge_error)
            )
        )

        line1 = ax1.errorbar(
            voltages_sorted,
            mean_ch_charge_sorted,
            yerr=mean_ch_charge_error_sorted,
            marker="o",
            capsize=4,
            label=f"{wafer_type} {sample_name}",
        )
        color = line1[0].get_color()

        ax2.errorbar(
            voltages_sorted,
            interpad_mean_charge_sorted,
            yerr=interpad_mean_charge_error_sorted,
            marker="o",
            capsize=4,
            label=f"{wafer_type} {sample_name}",
            color=color,
        )

    else:
        line_empty = ax1.errorbar([], [], yerr=[], marker="o", label=f"{wafer_type} {sample_name} Mean")
        color = line_empty[0].get_color()
        ax2.errorbar([], [], yerr=[], marker="o", label=f"{wafer_type} {sample_name}", color=color)

ax1.set_title("Mean impedance-charge of channels")
ax1.set_xlabel("Bias voltage, [V]")
ax1.set_ylabel(r"Mean impedance-charge, [mV$\cdot$ns]")
ax1.grid(True)
ax1.legend()

ax2.set_title("Inter-pad")
ax2.set_xlabel("Bias voltage, [V]")
ax2.set_ylabel(r"Mean impedance-charge, [mV$\cdot$ns]")
ax2.grid(True)
ax2.legend()

plt.tight_layout()

plt.savefig("plot.pdf", format="pdf", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
sample_list = [
    ["W2", "TW5_V1"],
    ["W11", "TW5_V2"],
    ["W9", "TW5_V1"],
    ["W5", "TW5_V1"],
    ["W11", "TW1_V2"],
    ["W11", "TW5_V1"],

    # ["W2","TW5_V1"],
    # ["W3","TW5_V1"],
    # ["W4","TW5_V1"],
    # ["W5","TW5_V1"],
    # ["W7","TW5_V1"],
    # ["W11","TW5_V1"],
    # ["W9","TW5_V1"],
]

percentage = 20

plt.figure(figsize=(10, 6), dpi=150)

for wafer_type, sample_name in sample_list:
    file_path = f"{data_folder}/{wafer_type}/{sample_name}/{sample_name}_150V.csv"
                
    df = read_analysis(file_path)

    borders, angle_deg = get_sensor_borders(df, threshold_fraction=0.7)

    cce_interpad_lines, cce_interpad_distance, cce_interpad_distance_error = analyze_cce(
        df, voltage, angle_deg=angle_deg, threshold=0.9, make_plot=False, normalization=True
    )
    
    cs_width, cs_width_error, cs_asymmetry, cs_interpad_lines = analyze_cs(
        df, voltage, angle_deg=angle_deg, threshold=0.9, make_plot=False
    )

    profile_df, jitter_width, jitter_width_error, jitter_asymmetry, jitter_interpad_lines = plot_jitter_profile(
        df,
        percentage=percentage,
        borders=borders,
        cce_interpad_lines=cce_interpad_lines,
        cs_interpad_lines=cs_interpad_lines,
        fit_options={
            "bins": 100,
            "left_lim": 98,
            "right_lim": 99.5,
            "peak_left": 98,
            "peak_right": 99.5,
        },
        make_plot=False,
    )

    y_min = profile_df["y"].min()
    y_max = profile_df["y"].max()
    y_range = y_max - y_min
    
    lower_bound = y_min + 0.30 * y_range
    upper_bound = y_min + 0.70 * y_range
    
    central_mask = (profile_df["y"] >= lower_bound) & (profile_df["y"] <= upper_bound)
    central_region = profile_df[central_mask]
    
    peak_idx = central_region["jitter_ps"].idxmax()
    peak_y = central_region.loc[peak_idx, "y"]

    profile_df["y"] = profile_df["y"] - peak_y

    plt.errorbar(
        profile_df["y"], 
        profile_df["jitter_ps"], 
        yerr=profile_df["jitter_err_ps"], 
        marker="o",
        markersize=4,
        linewidth=1,
        label=rf"{wafer_type} {sample_name} - {jitter_width:.2f} $\mu$m"
    )

plt.xlabel(r"y-coordinate, [$\mu$m]")
plt.ylabel(r"Jitter, [ps]")
plt.title("Jitter Profiles")

plt.grid(True)
plt.legend(title='Jitter width', loc="upper right", fontsize=9)
plt.tight_layout()


plt.savefig("plot.pdf", format="pdf", dpi=300, bbox_inches="tight")


plt.show()



In [ ]:
# voltages = [50, 60, 70, 80, 90, 100, 110, 120, 130, 140, 150]
# wafer_type = "W5"
# sample_names = ["TW5_V1", "TW5_V2", "TW1_V2", "TW2_V2"]
# percentage = 20

# for sample_name in sample_names:
#     for voltage in voltages:
#         file_path = f"{data_folder}/{wafer_type}/{sample_name}/{sample_name}_{voltage}V.csv"
        
#         if not os.path.exists(file_path):
#             print(f"Skipping {voltage}V: File not found.")
#             continue
            
#         print(f"Processing {sample_name} at {voltage}V...")
#         df = read_analysis(file_path)

#         borders, angle_deg = get_sensor_borders(df, threshold_fraction=0.7)

#         cce_interpad_lines, cce_interpad_distance, cce_interpad_distance_error = analyze_cce(
#             df, voltage, angle_deg=angle_deg, threshold=0.5, make_plot=False, normalization=True
#         )
        
#         csv_file = f"{data_folder}/{wafer_type}/{sample_name}/analyzed_data_50_percent.csv"
#         os.makedirs(os.path.dirname(csv_file), exist_ok=True)

#         new_row_df = pd.DataFrame([{
#             "voltage": voltage,
            
#             "cce_gap_distance": cce_interpad_distance,
#             "cce_gap_distance_error": cce_interpad_distance_error,    
#         }])

#         if os.path.isfile(csv_file):
#             existing_df = pd.read_csv(csv_file)
#             combined_df = pd.concat([existing_df, new_row_df], ignore_index=True)
#             combined_df.drop_duplicates(subset=['voltage'], keep='last', inplace=True)
#             combined_df.sort_values(by='voltage', inplace=True)
#             combined_df.to_csv(csv_file, index=False)
#         else:
#             new_row_df.to_csv(csv_file, index=False)

# print("Batch analysis complete!")

In [ ]:
sample_list = [
    # ["W11", "TW5_V1", []],
    # ["W11", "TW5_V2", []],
    # ["W11", "TW1_V2", []],
    # ["W11", "TW2_V2", []],

    # ["W2", "TW5_V1", [100]],
    # ["W2", "TW5_V2", []],
    # ["W2", "TW1_V2", []],
    # ["W2", "TW2_V2", []],

    ["W5", "TW5_V1", []],
    ["W5", "TW5_V2", []],
    ["W5", "TW1_V2", []],
    ["W5", "TW2_V2", []],
]

plt.figure(figsize=(10, 6))

for wafer_type, sample_name, exclusions in sample_list:
    file_path = f"{data_folder}/{wafer_type}/{sample_name}/analyzed_data_50_percent.csv"
    df = pd.read_csv(file_path)
    
    if exclusions:
        df = df[~df["voltage"].isin(exclusions)]

    voltages = df["voltage"].values
    cce_gaps = df["cce_gap_distance"].values
    cce_gap_errors = df["cce_gap_distance_error"].values

    if len(voltages) > 0:
        voltages_sorted, cce_gaps_sorted, cce_gap_errors_sorted = zip(
            *sorted(zip(voltages, cce_gaps, cce_gap_errors))
        )

        plt.errorbar(
            voltages_sorted,
            cce_gaps_sorted,
            yerr=cce_gap_errors_sorted,
            marker="o",
            capsize=4,
            label=f"{wafer_type} {sample_name}",
        )
    else:
        plt.plot([], [], marker="o", label=f"{wafer_type} {sample_name}")

plt.title("Charge Collection Efficiency (CCE) Gap Distance")
plt.xlabel("Bias voltage, [V]")
plt.ylabel(r"CCE Gap Distance, [$\mu$m]")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()